[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Biswajit1999/daily-astro-notebooks/blob/master/gaia/2026-07-30-proper-motion-membership-m67/notebook.ipynb)

# Finding M67 cluster members with a real proper-motion cut

**Learning goals** — after this notebook you'll be able to:
- Query a real Gaia DR3 field around an open cluster and build a proper-motion vector-point diagram.
- Numerically separate cluster members from field stars with a sigma-clip in proper-motion space (not by eye).
- Quantify the cluster's mean proper motion and its dispersion, with uncertainties.
- Validate the membership cut by showing the resulting color-magnitude diagram is a clean main sequence.

**Background.** Proper motion is how fast a star appears to creep across the sky each year (in milliarcseconds/yr). Stars born together in a cluster share almost the same motion through the galaxy, so they cluster tightly around one point in proper-motion space (a *vector-point diagram*), while foreground/background field stars scatter widely because they're at different distances and moving independently. M67 is an old (~4 Gyr), well-studied open cluster about 850-900 pc away, making it a good test case for a numeric proper-motion membership cut.

## 1. Query real Gaia DR3 data for the M67 field

In [ ]:
from astroquery.gaia import Gaia
import numpy as np
import matplotlib.pyplot as plt

ra_c, dec_c, radius_deg = 132.825, 11.814, 0.6

query_raw = f"""
SELECT TOP 6000 source_id, ra, dec, parallax, parallax_error, pmra, pmra_error,
       pmdec, pmdec_error, phot_g_mean_mag, bp_rp, ruwe
FROM gaiadr3.gaia_source
WHERE 1=CONTAINS(POINT('ICRS', ra, dec),
                  CIRCLE('ICRS', {ra_c}, {dec_c}, {radius_deg}))
  AND parallax BETWEEN -1 AND 4
  AND pmra IS NOT NULL AND pmdec IS NOT NULL
  AND phot_g_mean_mag < 18
"""
job_raw = Gaia.launch_job(query_raw)
tab_raw = job_raw.get_results()
n_raw = len(tab_raw)
print(f"Raw query returned {n_raw} sources within {radius_deg} deg of M67.")

pmra_all = np.array(tab_raw['pmra'])
pmdec_all = np.array(tab_raw['pmdec'])
g_all = np.array(tab_raw['phot_g_mean_mag'])
bprp_all = np.array(tab_raw['bp_rp'])

plt.figure(figsize=(6,6))
plt.scatter(pmra_all, pmdec_all, s=3, alpha=0.3, color='gray')
plt.xlim(-20,20); plt.ylim(-20,20)
plt.xlabel('pmRA (mas/yr)'); plt.ylabel('pmDec (mas/yr)')
plt.title('M67 field: all stars, proper-motion vector-point diagram')
plt.tight_layout()
plt.savefig('m67_pm_all.png', dpi=130)
plt.show()

## 2. Numeric membership cut: iterative sigma-clip around the cluster's motion

Rather than eyeballing the dense blob in the vector-point diagram, I run an iterative sigma-clip:
start from a rough box around M67's known bulk motion (roughly pmra=-10.97, pmdec=-2.94 mas/yr),
compute the mean and standard deviation of the points inside, then shrink the window to
3-sigma around that mean and repeat until it converges. This gives a reproducible, numeric
membership boundary instead of a hand-drawn one.

In [ ]:
pmra = pmra_all.copy()
pmdec = pmdec_all.copy()

# start near M67's known bulk proper motion
cx, cy = -10.97, -2.94
sel = (np.abs(pmra - cx) < 5) & (np.abs(pmdec - cy) < 5)

for iteration in range(6):
    mx, my = np.mean(pmra[sel]), np.mean(pmdec[sel])
    sx, sy = np.std(pmra[sel]), np.std(pmdec[sel])
    new_sel = (np.abs(pmra - mx) < 3*sx) & (np.abs(pmdec - my) < 3*sy)
    print(f"Iteration {iteration}: N={new_sel.sum()}, mean=({mx:.3f},{my:.3f}), std=({sx:.3f},{sy:.3f})")
    if new_sel.sum() == sel.sum():
        sel = new_sel
        break
    sel = new_sel

n_members = sel.sum()
cluster_pmra_mean, cluster_pmdec_mean = np.mean(pmra[sel]), np.mean(pmdec[sel])
cluster_pmra_std, cluster_pmdec_std = np.std(pmra[sel]), np.std(pmdec[sel])
pmra_err_mean = np.mean(np.array(tab_raw['pmra_error'])[sel])
pmdec_err_mean = np.mean(np.array(tab_raw['pmdec_error'])[sel])

print(f"\nConverged M67 candidate members: {n_members} of {n_raw} field stars ({100*n_members/n_raw:.1f}%)")
print(f"Cluster mean proper motion: pmRA = {cluster_pmra_mean:.3f} +/- {pmra_err_mean:.3f} mas/yr, "
      f"pmDec = {cluster_pmdec_mean:.3f} +/- {pmdec_err_mean:.3f} mas/yr")
print(f"Intrinsic dispersion (sigma-clip std): sigma_pmRA = {cluster_pmra_std:.3f} mas/yr, "
      f"sigma_pmDec = {cluster_pmdec_std:.3f} mas/yr")

plt.figure(figsize=(6,6))
plt.scatter(pmra_all, pmdec_all, s=3, alpha=0.15, color='gray', label='field')
plt.scatter(pmra[sel], pmdec[sel], s=6, alpha=0.6, color='crimson', label=f'members (N={n_members})')
plt.xlim(-20,20); plt.ylim(-20,20)
plt.xlabel('pmRA (mas/yr)'); plt.ylabel('pmDec (mas/yr)')
plt.title('M67: sigma-clipped members vs field')
plt.legend()
plt.tight_layout()
plt.savefig('m67_pm_members.png', dpi=130)
plt.show()

## 3. Validation: does the CMD of members look like a real cluster main sequence?

In [ ]:
parallax_m = np.array(tab_raw['parallax'])
good_member = sel & (parallax_m > 0)
good_field = (~sel) & (parallax_m > 0)

# subsample field stars for a fair-looking comparison plot
rng = np.random.default_rng(42)
field_idx = np.where(good_field)[0]
if len(field_idx) > 1500:
    field_idx = rng.choice(field_idx, 1500, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 6), sharey=True)
axes[0].scatter(bprp_all[field_idx], g_all[field_idx], s=5, alpha=0.4, color='gray')
axes[0].invert_yaxis()
axes[0].set_title(f'Field stars (random {len(field_idx)} shown)')
axes[0].set_xlabel('BP - RP'); axes[0].set_ylabel('Apparent G')

axes[1].scatter(bprp_all[good_member], g_all[good_member], s=8, alpha=0.6, color='crimson')
axes[1].invert_yaxis()
axes[1].set_title(f'Proper-motion members (N={good_member.sum()})')
axes[1].set_xlabel('BP - RP')
fig.suptitle('M67: field stars scatter broadly; PM-selected members trace a clean main sequence')
fig.tight_layout()
fig.savefig('m67_cmd_validation.png', dpi=130)
plt.show()

member_distance = 1000.0 / parallax_m[good_member]
print(f"Median distance of PM-selected members: {np.median(member_distance):.0f} pc "
      f"(literature M67 distance is roughly 850-900 pc)")

## What I'd look at next

- Add a parallax-consistency cut on top of the proper-motion cut (M67 members should also share nearly the same distance) to remove any remaining field interlopers that coincidentally share the cluster's proper motion.
- Fit the cleaned CMD with an isochrone to get a quantitative age estimate for M67.
- Compare the sigma-clip dispersion here to the *expected* dispersion from the individual `pmra_error`/`pmdec_error` values, to check how much of the scatter is measurement noise vs real intrinsic velocity dispersion in the cluster.

**Citation:** Data from Gaia DR3 (`gaiadr3.gaia_source`), ESA Gaia mission. See the Gaia credits page: https://www.cosmos.esa.int/web/gaia-users/credits